# 🏋️ Fine-Tune & Evaluate LightOnOCR-2-1B on Sinhala Synthetic Dataset

This notebook provides a complete, end-to-end pipeline to **fine-tune LightOnOCR-2-1B** using **LoRA (Low-Rank Adaptation)** on the [`Ransaka/sinhala_synthetic_ocr-large`](https://huggingface.co/datasets/Ransaka/sinhala_synthetic_ocr-large) dataset, followed by **in-depth benchmark evaluation** on a held-out test split.

### 🎯 Workflow Outline
1. **Environment Setup**: Lean, conflict-free Colab installation.
2. **Dataset Partitioning**: 90% Train / 10% Test split with fixed random seed (42).
3. **Model & LoRA Setup**: Load `LightOnOCR-2-1B-base` with target attention/MLP adapter layers.
4. **Multimodal Supervision**: Mask prompt & image tokens with `-100` so loss is computed strictly on Sinhala text.
5. **Training Loop**: Memory-efficient training with gradient accumulation and checkpoint saving.
6. **Post-Training Evaluation**: Evaluate the fine-tuned model on the unseen test set.
7. **Analysis & Metrics**: Corpus CER, Sample CER, WER, Exact Match %, Latency, Visual Error Analysis, and CSV export.

## 0. Installation & Dependencies
Run the cell below in Google Colab. We only install the packages Colab lacks (`transformers` from source, `peft`, `datasets`, `jiwer`), leaving pre-installed `pandas` and `numpy` untouched to avoid C-extension conflicts.

In [ ]:
# Install required packages for multimodal LoRA fine-tuning and OCR evaluation
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q git+https://github.com/huggingface/peft.git
!pip install -q datasets jiwer python-Levenshtein accelerate


## 1. Imports & GPU Setup
Verify hardware acceleration and configure deterministic seeds.

In [ ]:
import os
import time
import random
import unicodedata
from typing import List, Dict, Any, Tuple

import torch
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import jiwer
from datasets import load_dataset

from transformers import AutoProcessor, TrainingArguments, Trainer
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    try:
        from transformers import AutoModelForVision2Seq as AutoModelForImageTextToText
    except ImportError:
        from transformers import AutoModel as AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, PeftModel

# Set seeds for deterministic reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device & optimal precision configuration
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    device_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ Using CUDA: {device_name} ({gpu_mem:.2f} GB VRAM)")
    print(f"✅ Precision: {DTYPE}")
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    print("⚠️ CUDA not available. Running on CPU (training will be slow).")


## 2. Configuration & Hyperparameters
Customize training and evaluation settings. For a quick trial on Colab, you can set `max_train_samples = 500` or `1000`. To train on the entire synthetic dataset, set `max_train_samples = None`.

In [ ]:
class Config:
    """Fine-tuning and evaluation configuration."""
    # Model identifier (LightOn recommends starting from -base for custom fine-tuning)
    model_id: str = "lightonai/LightOnOCR-2-1B-base"
    
    # Dataset
    dataset_name: str = "Ransaka/sinhala_synthetic_ocr-large"
    test_split_ratio: float = 0.10  # 10% held-out test split
    
    # Sample limits (set to None to use full data, or integers for fast experimentation)
    max_train_samples: int | None = 1000   # e.g. 1000 for fast training, or None for all ~6,272
    max_eval_samples: int | None = 100     # e.g. 100 for fast test, or None for full ~697 test set
    
    # LoRA Hyperparameters
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    # Target all linear projection layers in the language model decoder
    target_modules: list = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    
    # Training Arguments
    output_dir: str = "./lightonocr_sinhala_lora"
    num_train_epochs: int = 1
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4   # Effective batch size = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    logging_steps: int = 10
    save_strategy: str = "epoch"
    
    # Generation & Evaluation
    max_new_tokens: int = 512
    eval_results_csv: str = "lightonocr_finetuned_eval_results.csv"

cfg = Config()
print(f"Model: {cfg.model_id}")
print(f"Target Train Samples: {cfg.max_train_samples if cfg.max_train_samples else 'ALL'}")
print(f"Target Eval Samples:  {cfg.max_eval_samples if cfg.max_eval_samples else 'ALL'}")
print(f"LoRA Rank: {cfg.lora_r}, Alpha: {cfg.lora_alpha}")

## 3. Load & Partition Dataset (Train / Test Split)
We split `Ransaka/sinhala_synthetic_ocr-large` into **90% Training** and a strictly held-out **10% Test Set**.

In [ ]:
print(f"Loading dataset '{cfg.dataset_name}'...")
raw_dataset = load_dataset(cfg.dataset_name, split="train")
total_rows = len(raw_dataset)
print(f"✅ Successfully loaded {total_rows:,} total pairs.")

# Detect image and text column names
image_col = next((c for c in ["image", "img"] if c in raw_dataset.column_names), None)
text_col = next((c for c in ["text", "label", "ground_truth", "transcription"] if c in raw_dataset.column_names), None)
print(f"Detected Image Column: '{image_col}', Text Column: '{text_col}'")

# Create deterministic Train / Test partition (90/10 split)
split_dataset = raw_dataset.train_test_split(test_size=cfg.test_split_ratio, seed=SEED)
train_data = split_dataset["train"]
test_data = split_dataset["test"]

# Subsample if configured
if cfg.max_train_samples is not None and cfg.max_train_samples < len(train_data):
    train_data = train_data.shuffle(seed=SEED).select(range(cfg.max_train_samples))
if cfg.max_eval_samples is not None and cfg.max_eval_samples < len(test_data):
    test_data = test_data.shuffle(seed=SEED).select(range(cfg.max_eval_samples))

print(f"✅ Final Train Samples: {len(train_data):,}")
print(f"✅ Final Held-out Test Samples: {len(test_data):,}")

### Preview Training Samples
Inspect 4 random training images alongside their Sinhala ground truth text.

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 5))
sample_indices = random.sample(range(len(train_data)), min(4, len(train_data)))

for ax, idx in zip(axes.flat, sample_indices):
    item = train_data[idx]
    img = item[image_col].convert("RGB")
    gt = item[text_col]
    ax.imshow(img)
    ax.set_title(f"Train #{idx}\nGT: {gt[:40]}..." if len(gt) > 40 else f"Train #{idx}\nGT: {gt}", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Load Base Model & Attach LoRA Adapter
We load `LightOnOCR-2-1B-base` and inject trainable Low-Rank Adaptation (LoRA) matrices into the attention and MLP projection layers. The vision encoder and remaining base weights are frozen.

In [ ]:
print(f"Loading processor and base model from '{cfg.model_id}'...")
processor = AutoProcessor.from_pretrained(cfg.model_id, trust_remote_code=True)
base_model = AutoModelForImageTextToText.from_pretrained(
    cfg.model_id,
    torch_dtype=DTYPE,
    trust_remote_code=True
).to(DEVICE)

# Configure LoRA (task_type=None for multimodal vision-language architectures)
peft_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=cfg.target_modules,
    bias="none"
)

# Wrap base model with LoRA adapter
model = get_peft_model(base_model, peft_config)
trainable_params, all_params = model.get_nb_trainable_parameters()

print("=" * 60)
print(f"✅ LoRA Adapter Attached Successfully!")
print(f"   Trainable Parameters: {trainable_params:,} ({trainable_params / all_params * 100:.2f}%)")
print(f"   Frozen Parameters:    {all_params - trainable_params:,}")
print(f"   Total Parameters:     {all_params:,}")
print("=" * 60)

## 5. Multimodal Data Collator with Loss Masking

To train an end-to-end OCR model properly:
1. We format each sample as a conversation:
   - **User**: Image (`{"type": "image", "image": img}`)
   - **Assistant**: Sinhala text (`{"type": "text", "text": gt_text}`)
2. We mask all user prompt and image tokens in `labels` with **`-100`**.
3. Cross-entropy loss is computed **only on the assistant's Sinhala output tokens**.

In [ ]:
class SinhalaOCRDataCollator:
    """
    Collate function that prepares multimodal inputs and masks user prompt tokens
    with -100 so the model is only supervised on predicting Sinhala text.
    """
    def __init__(self, processor: Any, max_new_tokens: int = 512):
        self.processor = processor
        self.max_new_tokens = max_new_tokens
        
    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids_list = []
        labels_list = []
        pixel_values_list = []
        
        for item in batch:
            img = item[image_col]
            if img.mode != "RGB":
                img = img.convert("RGB")
            gt_text = str(item[text_col]) if item[text_col] is not None else ""
            
            # 1. Conversation without assistant response (to measure prompt token count)
            prompt_conv = [
                {
                    "role": "user",
                    "content": [{"type": "image", "image": img}]
                }
            ]
            prompt_inputs = self.processor.apply_chat_template(
                prompt_conv,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt"
            )
            prompt_len = prompt_inputs["input_ids"].shape[1]
            
            # 2. Full conversation with assistant response (for training)
            full_conv = [
                {
                    "role": "user",
                    "content": [{"type": "image", "image": img}]
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": gt_text}]
                }
            ]
            full_inputs = self.processor.apply_chat_template(
                full_conv,
                add_generation_prompt=False,
                tokenize=True,
                return_dict=True,
                return_tensors="pt"
            )
            
            # 3. Create labels: copy input_ids and mask prompt tokens with -100
            labels = full_inputs["input_ids"].clone()
            labels[:, :prompt_len] = -100
            
            input_ids_list.append(full_inputs["input_ids"].squeeze(0))
            labels_list.append(labels.squeeze(0))
            if "pixel_values" in full_inputs:
                pixel_values_list.append(full_inputs["pixel_values"].squeeze(0))
                
        # Pad input_ids and labels to max sequence length in batch
        pad_token_id = self.processor.tokenizer.pad_token_id if hasattr(self.processor, "tokenizer") and self.processor.tokenizer.pad_token_id is not None else 0
        
        padded_input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids_list, batch_first=True, padding_value=pad_token_id
        )
        padded_labels = torch.nn.utils.rnn.pad_sequence(
            labels_list, batch_first=True, padding_value=-100
        )
        attention_mask = (padded_input_ids != pad_token_id).long()
        
        collated = {
            "input_ids": padded_input_ids,
            "attention_mask": attention_mask,
            "labels": padded_labels,
        }
        if pixel_values_list:
            collated["pixel_values"] = torch.stack(pixel_values_list)
            
        return collated

collator = SinhalaOCRDataCollator(processor=processor)
print("✅ Multimodal Data Collator ready.")

## 6. Execute Fine-Tuning
Run training using Hugging Face `Trainer` with gradient accumulation and automatic mixed precision.

In [ ]:
training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    logging_steps=cfg.logging_steps,
    save_strategy=cfg.save_strategy,
    save_total_limit=1,
    fp16=(DTYPE == torch.float16),
    bf16=(DTYPE == torch.bfloat16),
    remove_unused_columns=False,  # Essential for custom multimodal datasets
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    data_collator=collator
)

print("🚀 Starting fine-tuning...")
train_start_time = time.time()
train_result = trainer.train()
total_train_time = time.time() - train_start_time

print(f"\n✅ Training completed in {total_train_time / 60:.2f} minutes!")
print(f"   Final Loss: {train_result.training_loss:.4f}")

# Save LoRA adapter weights and processor
print(f"Saving fine-tuned adapter to '{cfg.output_dir}'...")
model.save_pretrained(cfg.output_dir)
processor.save_pretrained(cfg.output_dir)
print(f"💾 Adapter weights saved to: {cfg.output_dir}")

## 7. Post-Training Evaluation Setup
We now evaluate the fine-tuned model on the **held-out 10% test split** (`test_data`) using the exact same metrics as our baseline benchmark:
- **Unicode NFC Normalization** for Sinhala combining diacritics
- **Character Error Rate (CER)**
- **Word Error Rate (WER)**
- **Exact Match (%)**

In [ ]:
def normalize_sinhala_text(text: str) -> str:
    """Normalizes Sinhala text using Unicode NFC and collapses whitespace."""
    if not text:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    return " ".join(text.split())

def compute_ocr_metrics(reference: str, hypothesis: str) -> Dict[str, float]:
    """Computes CER, WER, and Exact Match between reference and hypothesis."""
    norm_ref = normalize_sinhala_text(reference)
    norm_hyp = normalize_sinhala_text(hypothesis)
    exact_match = 1.0 if norm_ref == norm_hyp else 0.0
    
    if len(norm_ref) == 0:
        cer = 0.0 if len(norm_hyp) == 0 else 1.0
        wer = 0.0 if len(norm_hyp) == 0 else 1.0
        return {"cer": cer, "wer": wer, "exact_match": exact_match}
        
    cer = float(jiwer.cer(norm_ref, norm_hyp))
    words_ref = norm_ref.split()
    wer = cer if len(words_ref) == 0 else float(jiwer.wer(norm_ref, norm_hyp))
    
    return {"cer": cer, "wer": wer, "exact_match": exact_match}

def extract_sinhala_text(image: Image.Image, max_new_tokens: int = 512) -> Tuple[str, float]:
    """Performs inference with the fine-tuned model on an image."""
    if image.mode != "RGB":
        image = image.convert("RGB")
        
    conv = [{"role": "user", "content": [{"type": "image", "image": image}]}]
    start_t = time.perf_counter()
    
    inputs = processor.apply_chat_template(
        conv,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )
    model_inputs = {
        k: v.to(device=DEVICE, dtype=DTYPE) if v.is_floating_point() else v.to(device=DEVICE)
        for k, v in inputs.items()
    }
    
    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
        
    prompt_len = model_inputs["input_ids"].shape[1]
    pred = processor.decode(output_ids[0, prompt_len:], skip_special_tokens=True).strip()
    latency = time.perf_counter() - start_t
    return pred, latency

model.eval()
print("✅ Model set to eval() mode for benchmark evaluation.")

## 8. Run Benchmark on Held-out Test Split
We evaluate the fine-tuned model on the unseen test split and log latency and error rates.

In [ ]:
eval_records: List[Dict[str, Any]] = []
print(f"Starting evaluation on {len(test_data)} held-out test samples...")

for idx in tqdm(range(len(test_data)), desc="Evaluating Fine-Tuned LightOnOCR"):
    item = test_data[idx]
    img = item[image_col]
    gt_text = str(item[text_col]) if item[text_col] is not None else ""
    
    try:
        pred_text, latency = extract_sinhala_text(img, max_new_tokens=cfg.max_new_tokens)
        metrics = compute_ocr_metrics(reference=gt_text, hypothesis=pred_text)
        eval_records.append({
            "sample_id": idx,
            "ground_truth": gt_text,
            "predicted_text": pred_text,
            "gt_char_len": len(normalize_sinhala_text(gt_text)),
            "pred_char_len": len(normalize_sinhala_text(pred_text)),
            "cer": metrics["cer"],
            "wer": metrics["wer"],
            "exact_match": metrics["exact_match"],
            "latency_sec": latency,
            "error": None
        })
    except Exception as e:
        eval_records.append({
            "sample_id": idx,
            "ground_truth": gt_text,
            "predicted_text": "",
            "gt_char_len": len(normalize_sinhala_text(gt_text)),
            "pred_char_len": 0,
            "cer": 1.0,
            "wer": 1.0,
            "exact_match": 0.0,
            "latency_sec": 0.0,
            "error": str(e)
        })

df_finetuned = pd.DataFrame(eval_records)
print(f"✅ Evaluation complete across {len(df_finetuned)} test samples.")

## 9. Fine-Tuned Benchmark Results
Aggregate metrics on the held-out test split.

In [ ]:
mean_cer = df_finetuned["cer"].mean() * 100
median_cer = df_finetuned["cer"].median() * 100
mean_wer = df_finetuned["wer"].mean() * 100
median_wer = df_finetuned["wer"].median() * 100
exact_match_rate = df_finetuned["exact_match"].mean() * 100
avg_latency_ms = df_finetuned["latency_sec"].mean() * 1000

all_refs = [normalize_sinhala_text(r) for r in df_finetuned["ground_truth"]]
all_preds = [normalize_sinhala_text(p) for p in df_finetuned["predicted_text"]]
corpus_cer = jiwer.cer(all_refs, all_preds) * 100
corpus_wer = jiwer.wer(all_refs, all_preds) * 100

summary_data = {
    "Metric": [
        "Corpus CER (%)",
        "Mean Sample CER (%)",
        "Median Sample CER (%)",
        "Corpus WER (%)",
        "Mean Sample WER (%)",
        "Median Sample WER (%)",
        "Exact Match Accuracy (%)",
        "Avg Latency per Sample (ms)",
        "Test Samples Evaluated"
    ],
    "Fine-Tuned Result": [
        f"{corpus_cer:.2f}%",
        f"{mean_cer:.2f}%",
        f"{median_cer:.2f}%",
        f"{corpus_wer:.2f}%",
        f"{mean_wer:.2f}%",
        f"{median_wer:.2f}%",
        f"{exact_match_rate:.2f}%",
        f"{avg_latency_ms:.1f} ms",
        f"{len(df_finetuned)}"
    ]
}

df_summary = pd.DataFrame(summary_data)
print("=" * 60)
print(f"   FINE-TUNED ACCURACY BENCHMARK (Held-out Test Split)")
print("=" * 60)
print(df_summary.to_string(index=False))
print("=" * 60)

# Save to CSV
df_finetuned.to_csv(cfg.eval_results_csv, index=False, encoding="utf-8-sig")
print(f"💾 Detailed predictions saved to: {cfg.eval_results_csv}")

## 10. Error Distribution Visualizations
Visualize the distribution of CER and WER across test samples.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# CER Distribution
sns.histplot(df_finetuned["cer"] * 100, bins=25, kde=True, ax=axes[0], color="royalblue")
axes[0].axvline(mean_cer, color="red", linestyle="--", label=f"Mean CER: {mean_cer:.1f}%")
axes[0].set_title("Fine-Tuned CER Distribution (%)", fontsize=12)
axes[0].set_xlabel("CER (%)")
axes[0].set_ylabel("Count")
axes[0].legend()

# WER Distribution
sns.histplot(df_finetuned["wer"] * 100, bins=25, kde=True, ax=axes[1], color="coral")
axes[1].axvline(mean_wer, color="darkred", linestyle="--", label=f"Mean WER: {mean_wer:.1f}%")
axes[1].set_title("Fine-Tuned WER Distribution (%)", fontsize=12)
axes[1].set_xlabel("WER (%)")
axes[1].set_ylabel("Count")
axes[1].legend()

# CER vs Character Count
sns.scatterplot(data=df_finetuned, x="gt_char_len", y=df_finetuned["cer"] * 100, ax=axes[2], alpha=0.6, color="purple")
axes[2].set_title("CER vs Ground Truth Length", fontsize=12)
axes[2].set_xlabel("Character Count")
axes[2].set_ylabel("CER (%)")

plt.tight_layout()
plt.show()

## 11. Qualitative Gallery: Best & Challenging Predictions
Inspect the model's actual predictions on held-out test images.

In [ ]:
def display_case_gallery(sample_indices: List[int], title: str):
    print(f"\n{'=' * 20} {title} {'=' * 20}")
    for idx in sample_indices:
        row = df_finetuned[df_finetuned["sample_id"] == idx].iloc[0]
        img = test_data[idx][image_col].convert("RGB")
        
        plt.figure(figsize=(10, 2.5))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Test Sample #{idx} | CER: {row['cer']*100:.1f}% | WER: {row['wer']*100:.1f}% | Match: {bool(row['exact_match'])}")
        plt.show()
        
        print(f"[📝 Ground Truth]: {row['ground_truth']}")
        print(f"[🤖 Model Pred  ]: {row['predicted_text']}")
        print("-" * 80)

# Top 3 Best Predictions
best_indices = df_finetuned.sort_values("cer", ascending=True)["sample_id"].head(3).tolist()
display_case_gallery(best_indices, "TOP 3 BEST PREDICTIONS (Lowest CER)")

# Top 3 Challenging Predictions (Highest CER)
worst_indices = df_finetuned.sort_values("cer", ascending=False)["sample_id"].head(3).tolist()
display_case_gallery(worst_indices, "TOP 3 CHALLENGING PREDICTIONS (Highest CER)")

## 12. Save & Download LoRA Weights from Colab
Run the cell below to zip the saved LoRA adapter folder and download it directly from Google Colab.

In [ ]:
# Zip and download the LoRA adapter folder
import shutil
shutil.make_archive("lightonocr_sinhala_lora", "zip", cfg.output_dir)
print(f"✅ LoRA weights zipped to 'lightonocr_sinhala_lora.zip'")

# Download in Google Colab
try:
    from google.colab import files
    files.download("lightonocr_sinhala_lora.zip")
    files.download(cfg.eval_results_csv)
    print("⬇️ Download triggered in Colab.")
except ImportError:
    print("ℹ️ Running outside Colab. Files are saved in local directory.")